# 11 - Efficiency Metrics: Parameter Count, FLOPs, and Inference Latency

**Project:** Regime-Conditional Attention Head Selection for Time Series Transformers (ReCAHS)

The project proposal asks for parameter count, FLOPs, and inference latency alongside forecasting accuracy for every pruning method. None of that has been measured yet, for an important reason: every experiment so far (04-10) implements pruning as **soft/functional masking** — `HeadMaskController` still computes every head's `query`/`key`/`value` projections and only zeroes out the pruned ones' contribution afterward. This is the right tool for *searching* over candidate masks (cheap to try, no architecture surgery needed), but it means all those models are, architecturally, still the full 24-head B4 model — measuring parameters/FLOPs/latency on them directly would show **zero** difference from the unpruned baseline, which would be a misleading answer to what the proposal is actually asking.

This notebook instead performs **structural pruning**: it physically removes the pruned heads' rows/columns from each layer's `query_projection` / `key_projection` / `value_projection` / `out_projection` weight matrices, producing genuinely smaller models, and verifies each one produces (up to floating-point rounding) the same output as the original soft-masked model before measuring:

1. **Parameter count** — direct count of the structurally reduced model's parameters.
2. **FLOPs** — via `thop`, on a representative single-sample input (Note: `thop` hooks standard layers like `nn.Linear`; custom attention math implemented with raw tensor ops may be under-counted — treat these numbers as a consistent *relative* comparison across methods rather than an absolute FLOPs figure).
3. **Inference latency** — wall-clock GPU time for a realistic batch (32), averaged over many repeats after a warm-up period.

**Which methods get structurally pruned.** Static pruning (Experiment 04) uses one fixed mask, so it becomes one smaller model. The joint dynamic method (Experiment 10, the best-performing method) uses a *different* 18-head subset per regime, so it becomes **three** smaller models (trend/seasonal/residual) that a deployment would switch between based on the detected regime of each input window; this notebook also reports a regime-frequency-weighted "expected" cost for Policy B using the test-set regime distribution from Experiment 07/10, plus the extra wall-clock cost of the STL regime-detection step itself (a real cost Policy B pays that Policy A does not). Random pruning, magnitude-based pruning, and independent-scoring dynamic pruning all remove the same *count* of heads (6 of 24) as static pruning, just different specific heads — their parameter/FLOPs savings are the same order of magnitude as static's and are not re-measured separately here; only their forecasting accuracy (already reported in Experiments 08/10) differs.


## 1. Mount Google Drive and import core libraries

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import sys
import os
import shutil
import copy
import time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from tqdm.auto import tqdm

## 2. Define project paths

In [3]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/BIL401_Regime_Head_Pruning"
)

REGIME_DIR = PROJECT_DIR / "regime_detection"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
HEAD_IMPORTANCE_DIR = PROJECT_DIR / "head_importance"
PRUNING_DIR = PROJECT_DIR / "pruning_experiments"
JOINT_DIR = PRUNING_DIR / "b4_joint_dynamic_75_keep"
EFFICIENCY_DIR = PRUNING_DIR / "b4_efficiency_metrics"

EFFICIENCY_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("JOINT_DIR:", JOINT_DIR)
print("EFFICIENCY_DIR:", EFFICIENCY_DIR)

PROJECT_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning
JOINT_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/b4_joint_dynamic_75_keep
EFFICIENCY_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/b4_efficiency_metrics


## 3. Set up Time-Series-Library

In [4]:
TSLIB_DIR = Path("/content/Time-Series-Library")

if not TSLIB_DIR.exists():
    %cd /content
    !git clone https://github.com/thuml/Time-Series-Library.git
else:
    print("Time-Series-Library already exists:", TSLIB_DIR)

sys.path.insert(0, str(TSLIB_DIR))

/content
Cloning into 'Time-Series-Library'...
remote: Enumerating objects: 2295, done.
remote: Total 2295 (delta 0), reused 0 (delta 0), pack-reused 2295 (from 1)
Receiving objects: 100% (2295/2295), 78.43 MiB | 34.57 MiB/s, done.
Resolving deltas: 100% (1570/1570), done.


In [5]:
%cd /content/Time-Series-Library
!pip install -q patool sktime scikit-base statsmodels thop --no-deps
!pip install -q --no-deps einops
!pip install reformer-pytorch --no-deps
!pip install local-attention --no-deps
!pip install hyper_connections --no-deps
!pip install axial_positional_embedding --no-deps
!pip install product_key_memory --no-deps
!pip install colt5_attention --no-deps

/content/Time-Series-Library
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 17.2 MB/s eta 0:00:00


## 4. Download the ETTh1 dataset

In [6]:
drive_data_path = PROJECT_DIR / "data" / "ETTh1.csv"

tslib_data_path = (
    TSLIB_DIR
    / "dataset/ETDataset/ETT-small/ETTh1.csv"
)

tslib_data_path.parent.mkdir(parents=True, exist_ok=True)

if drive_data_path.exists():
    shutil.copy2(drive_data_path, tslib_data_path)
    print("ETTh1 copied from Drive.")
else:
    print("Drive data not found. Downloading ETTh1...")
    !wget -q https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv -O /content/Time-Series-Library/dataset/ETDataset/ETT-small/ETTh1.csv

print("Dataset exists:", tslib_data_path.exists())

Drive data not found. Downloading ETTh1...
Dataset exists: True


## 5. Load pruning masks

The static 25% pruning list (committed to the repository) and the three joint dynamic masks saved by notebook 10 (on Drive, under `pruning_experiments/b4_joint_dynamic_75_keep/`).

In [7]:
static_prune_df = pd.read_csv(HEAD_IMPORTANCE_DIR / "summaries" / "static_prune_25_percent_heads.csv")
print("Static pruned (layer, head) pairs:")
display(static_prune_df[["layer", "head"]])

joint_masks_raw = {}
for regime in ["trend", "seasonal", "residual"]:
    mask_path = JOINT_DIR / f"{regime}_joint_dynamic_keep_75_mask.csv"
    mask_df = pd.read_csv(mask_path, index_col=0)
    joint_masks_raw[regime] = torch.tensor(mask_df.values, dtype=torch.float32)
    print(f"{regime}: loaded mask with {int(joint_masks_raw[regime].sum().item())} active heads")

# Test-set regime frequency (from Experiment 07 / notebook 10), used to weight Policy B's expected cost
TEST_REGIME_FREQUENCY = {"trend": 2606 / 2785, "seasonal": 125 / 2785, "residual": 54 / 2785}
print("\nTest regime frequency:", TEST_REGIME_FREQUENCY)

Static pruned (layer, head) pairs:


,layer,head
0,0,5
1,1,7
2,2,0
3,1,4
4,1,1
5,0,3


trend: loaded mask with 18 active heads
seasonal: loaded mask with 18 active heads
residual: loaded mask with 18 active heads

Test regime frequency: {'trend': 0.9357271095152603, 'seasonal': 0.04488330341113106, 'residual': 0.01938958707360862}


## 6. Locate the B4 checkpoint

In [8]:
b4_checkpoint_candidates = list(
    CHECKPOINT_DIR.glob("B4_patchtst_etth1_336_dm128_h8/**/checkpoint.pth")
)

if len(b4_checkpoint_candidates) == 0:
    raise FileNotFoundError("B4 checkpoint bulunamadı. Run notebook 00 first if needed.")

b4_checkpoint_path = b4_checkpoint_candidates[0]
print("Selected checkpoint:", b4_checkpoint_path)

Selected checkpoint: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints/B4_patchtst_etth1_336_dm128_h8/checkpoint.pth


## 7. Reconstruct the B4 model arguments

In [9]:
from argparse import Namespace

args = Namespace(
    task_name="long_term_forecast",
    is_training=0,
    model_id="ETTh1_336_96_dm128_h8",
    model="PatchTST",

    data="ETTh1",
    root_path="./dataset/ETDataset/ETT-small/",
    data_path="ETTh1.csv",
    features="M",
    target="OT",
    freq="h",
    checkpoints="./checkpoints/",

    seq_len=336,
    label_len=48,
    pred_len=96,
    seasonal_patterns="Monthly",
    inverse=False,

    enc_in=7,
    dec_in=7,
    c_out=7,
    d_model=128,
    n_heads=8,
    e_layers=3,
    d_layers=1,
    d_ff=256,
    moving_avg=25,
    factor=3,
    distil=True,
    dropout=0.1,
    embed="timeF",
    activation="gelu",
    output_attention=False,

    patch_len=16,
    stride=8,
    padding_patch="end",
    revin=1,
    affine=0,
    subtract_last=0,
    decomposition=0,
    kernel_size=25,
    individual=0,

    num_workers=0,
    itr=1,
    train_epochs=10,
    batch_size=32,
    patience=3,
    learning_rate=0.0001,
    des="baseline_b4",
    loss="MSE",
    lradj="type1",
    use_amp=False,
    augmentation_ratio=0.0,

    use_gpu=torch.cuda.is_available(),
    gpu=0,
    use_multi_gpu=False,
    devices="0",
    gpu_type="cuda",
    expand=2,
    d_conv=4,
    top_k=5,
    num_kernels=6,
    channel_independence=0,
    decomp_method="moving_avg",
    use_norm=1,
    down_sampling_layers=0,
    down_sampling_window=1,
    down_sampling_method=None,
    seg_len=48,

    p_hidden_dims=[128, 128],
    p_hidden_layers=2,
)

print(args)

Namespace(task_name='long_term_forecast', is_training=0, model_id='ETTh1_336_96_dm128_h8', model='PatchTST', data='ETTh1', root_path='./dataset/ETDataset/ETT-small/', data_path='ETTh1.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=336, label_len=48, pred_len=96, seasonal_patterns='Monthly', inverse=False, enc_in=7, dec_in=7, c_out=7, d_model=128, n_heads=8, e_layers=3, d_layers=1, d_ff=256, moving_avg=25, factor=3, distil=True, dropout=0.1, embed='timeF', activation='gelu', output_attention=False, patch_len=16, stride=8, padding_patch='end', revin=1, affine=0, subtract_last=0, decomposition=0, kernel_size=25, individual=0, num_workers=0, itr=1, train_epochs=10, batch_size=32, patience=3, learning_rate=0.0001, des='baseline_b4', loss='MSE', lradj='type1', use_amp=False, augmentation_ratio=0.0, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0', gpu_type='cuda', expand=2, d_conv=4, top_k=5, num_kernels=6, channel_independence=0, decomp_method='moving_

## 8. Load the baseline (unpruned) model

In [10]:
from exp.exp_long_term_forecasting import Exp_Long_Term_Forecast

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def load_fresh_b4_model():
    exp = Exp_Long_Term_Forecast(args)
    m = exp.model.to(device)
    state = torch.load(b4_checkpoint_path, map_location=device)
    m.load_state_dict(state)
    m.eval()
    return m

baseline_model = load_fresh_b4_model()
print("Baseline B4 model loaded.")

num_layers = len(baseline_model.encoder.attn_layers)
num_heads = baseline_model.encoder.attn_layers[0].attention.n_heads
print("Layers:", num_layers, "| heads/layer:", num_heads)

Device: cuda:0
Use GPU: cuda:0
🚀 Lazy Loading: PatchTST ...
Baseline B4 model loaded.
Layers: 3 | heads/layer: 8


## 9. Build the validation loader (for correctness checks and latency benchmarking)

In [11]:
from data_provider.data_factory import data_provider

vali_data, vali_loader = data_provider(args, flag="val")
print("Validation batches:", len(vali_loader))

sample_batch = next(iter(vali_loader))
sample_x, sample_y, sample_x_mark, sample_y_mark = sample_batch
sample_x = sample_x.float().to(device)
sample_y = sample_y.float().to(device)
sample_x_mark = sample_x_mark.float().to(device)
sample_y_mark = sample_y_mark.float().to(device)

print("Sample batch shapes:", sample_x.shape, sample_x_mark.shape, sample_y.shape, sample_y_mark.shape)

val 2785
Validation batches: 88
Sample batch shapes: torch.Size([32, 336, 7]) torch.Size([32, 336, 4]) torch.Size([32, 144, 7]) torch.Size([32, 144, 4])


## 10. Structural pruning: physically remove pruned heads' weights

For a given per-layer list of *heads to keep*, this rebuilds each attention layer's four projection matrices with only the kept heads' rows/columns, and updates `attention_layer.n_heads` accordingly. `inner_attention` (the scaled dot-product mechanism) reads `n_heads` dynamically from the layer at call time (the same property `HeadMaskController` relies on in notebooks 04-10), so it works unmodified with a smaller head count.

In [12]:
def structurally_prune_attention_layer(attention_layer, keep_heads):
    device_ = attention_layer.query_projection.weight.device
    dtype_ = attention_layer.query_projection.weight.dtype

    d_model = attention_layer.query_projection.in_features
    n_heads_original = attention_layer.n_heads
    d_keys = attention_layer.query_projection.out_features // n_heads_original
    d_values = attention_layer.value_projection.out_features // n_heads_original
    new_n_heads = len(keep_heads)

    def sliced_linear_rows(linear, d_per_head, out_features):
        rows = [linear.weight.data[h * d_per_head:(h + 1) * d_per_head, :] for h in keep_heads]
        new_weight = torch.cat(rows, dim=0)
        new_linear = nn.Linear(linear.in_features, out_features, bias=linear.bias is not None)
        new_linear.weight.data = new_weight.clone()
        if linear.bias is not None:
            bias_parts = [linear.bias.data[h * d_per_head:(h + 1) * d_per_head] for h in keep_heads]
            new_linear.bias.data = torch.cat(bias_parts, dim=0)
        return new_linear.to(device=device_, dtype=dtype_)

    new_q = sliced_linear_rows(attention_layer.query_projection, d_keys, d_keys * new_n_heads)
    new_k = sliced_linear_rows(attention_layer.key_projection, d_keys, d_keys * new_n_heads)
    new_v = sliced_linear_rows(attention_layer.value_projection, d_values, d_values * new_n_heads)

    out_cols = [attention_layer.out_projection.weight.data[:, h * d_values:(h + 1) * d_values] for h in keep_heads]
    new_out_weight = torch.cat(out_cols, dim=1)
    new_out = nn.Linear(d_values * new_n_heads, d_model, bias=attention_layer.out_projection.bias is not None)
    new_out.weight.data = new_out_weight.clone()
    if attention_layer.out_projection.bias is not None:
        new_out.bias.data = attention_layer.out_projection.bias.data.clone()
    new_out = new_out.to(device=device_, dtype=dtype_)

    attention_layer.query_projection = new_q
    attention_layer.key_projection = new_k
    attention_layer.value_projection = new_v
    attention_layer.out_projection = new_out
    attention_layer.n_heads = new_n_heads


def mask_to_keep_heads_per_layer(mask, num_layers, num_heads):
    return [
        sorted(h for h in range(num_heads) if mask[l, h].item() == 1.0)
        for l in range(num_layers)
    ]


def build_structurally_pruned_model(keep_heads_per_layer):
    pruned_model = load_fresh_b4_model()
    for layer_idx, encoder_layer in enumerate(pruned_model.encoder.attn_layers):
        structurally_prune_attention_layer(encoder_layer.attention, keep_heads_per_layer[layer_idx])
    pruned_model.eval()
    return pruned_model


## 11. Build all structurally pruned model variants

In [13]:
static_mask = torch.ones(num_layers, num_heads)
for _, row in static_prune_df.iterrows():
    static_mask[int(row["layer"]), int(row["head"])] = 0.0

keep_heads_static = mask_to_keep_heads_per_layer(static_mask, num_layers, num_heads)
keep_heads_joint = {
    regime: mask_to_keep_heads_per_layer(joint_masks_raw[regime], num_layers, num_heads)
    for regime in ["trend", "seasonal", "residual"]
}

print("Static keep-heads per layer:", keep_heads_static)
for regime, kh in keep_heads_joint.items():
    print(f"{regime} keep-heads per layer:", kh)

pruned_models = {"static_25": build_structurally_pruned_model(keep_heads_static)}
for regime in ["trend", "seasonal", "residual"]:
    pruned_models[f"joint_{regime}"] = build_structurally_pruned_model(keep_heads_joint[regime])

print("\nBuilt structurally pruned models:", list(pruned_models.keys()))

Static keep-heads per layer: [[0, 1, 2, 4, 6, 7], [0, 2, 3, 5, 6], [1, 2, 3, 4, 5, 6, 7]]
trend keep-heads per layer: [[0, 1, 2, 4, 5, 7], [0, 1, 3, 5, 6], [1, 2, 3, 4, 5, 6, 7]]
seasonal keep-heads per layer: [[0, 1, 2, 6, 7], [0, 1, 2, 3, 4, 5, 7], [0, 1, 2, 4, 6, 7]]
residual keep-heads per layer: [[0, 1, 2, 4, 7], [0, 1, 2, 3, 5, 6], [0, 1, 2, 3, 4, 6, 7]]
Use GPU: cuda:0
🚀 Lazy Loading: PatchTST ...
Use GPU: cuda:0
🚀 Lazy Loading: PatchTST ...
Use GPU: cuda:0
🚀 Lazy Loading: PatchTST ...
Use GPU: cuda:0
🚀 Lazy Loading: PatchTST ...

Built structurally pruned models: ['static_25', 'joint_trend', 'joint_seasonal', 'joint_residual']


## 12. Correctness check: structurally pruned output matches soft-masked output

Reuses the same `HeadMaskController` pattern from notebooks 04-09 (single global mask) on a fresh copy of the baseline model, and compares its masked output against the corresponding structurally pruned model's output on the same input batch. Any difference should be at floating-point rounding level only — the two are mathematically the same computation, just with the always-zero terms physically removed in the structural version.

In [14]:
class HeadMaskController:
    def __init__(self, model):
        self.model = model
        self.original_forwards = {}
        self.current_mask = None

    def install(self):
        for layer_idx, encoder_layer in enumerate(self.model.encoder.attn_layers):
            attention_layer = encoder_layer.attention
            if layer_idx in self.original_forwards:
                continue
            original_forward = attention_layer.forward
            self.original_forwards[layer_idx] = original_forward

            def make_masked_forward(layer_idx, attention_layer):
                def masked_forward(queries, keys, values, attn_mask, tau=None, delta=None):
                    B, L, _ = queries.shape
                    _, S, _ = keys.shape
                    H = attention_layer.n_heads

                    q = attention_layer.query_projection(queries).view(B, L, H, -1)
                    k = attention_layer.key_projection(keys).view(B, S, H, -1)
                    v = attention_layer.value_projection(values).view(B, S, H, -1)

                    out, attn = attention_layer.inner_attention(q, k, v, attn_mask, tau=tau, delta=delta)

                    if self.current_mask is not None:
                        layer_mask = self.current_mask[layer_idx].to(out.device).view(1, 1, H, 1)
                        out = out * layer_mask

                    out = out.view(B, L, -1)
                    return attention_layer.out_projection(out), attn

                return masked_forward

            attention_layer.forward = make_masked_forward(layer_idx, attention_layer)

    def set_mask(self, mask):
        self.current_mask = mask.clone().float()


reference_model = load_fresh_b4_model()
reference_controller = HeadMaskController(reference_model)
reference_controller.install()

masks_to_check = {"static_25": static_mask, **{f"joint_{r}": joint_masks_raw[r] for r in ["trend", "seasonal", "residual"]}}

with torch.no_grad():
    for name, mask in masks_to_check.items():
        reference_controller.set_mask(mask)
        soft_output = reference_model(sample_x, sample_x_mark, sample_y, sample_y_mark)
        hard_output = pruned_models[name](sample_x, sample_x_mark, sample_y, sample_y_mark)

        max_diff = torch.max(torch.abs(soft_output - hard_output)).item()
        print(f"{name}: max abs difference between soft-masked and structurally-pruned outputs = {max_diff:.8f}")
        assert max_diff < 1e-3, f"{name}: structural pruning does not match soft masking!"

print("\nAll structurally pruned models match their soft-masked equivalents.")

Use GPU: cuda:0
🚀 Lazy Loading: PatchTST ...
static_25: max abs difference between soft-masked and structurally-pruned outputs = 0.00000000
joint_trend: max abs difference between soft-masked and structurally-pruned outputs = 0.00000000
joint_seasonal: max abs difference between soft-masked and structurally-pruned outputs = 0.00000000
joint_residual: max abs difference between soft-masked and structurally-pruned outputs = 0.00000000

All structurally pruned models match their soft-masked equivalents.


## 13. Parameter counts

In [15]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

param_counts = {"B4_no_pruning": count_parameters(baseline_model)}
for name, m in pruned_models.items():
    param_counts[name] = count_parameters(m)

param_df = pd.DataFrame([
    {"setting": k, "parameters": v, "parameters_vs_baseline_percent": (v / param_counts["B4_no_pruning"] - 1) * 100}
    for k, v in param_counts.items()
])
display(param_df)

,setting,parameters,parameters_vs_baseline_percent
0,B4_no_pruning,915936,0.000000
1,static_25,866496,-5.397757
2,joint_trend,866496,-5.397757
3,joint_seasonal,866496,-5.397757
4,joint_residual,866496,-5.397757


## 14. FLOPs (via `thop`)

Measured on a single-sample input (`batch_size=1`), which is `thop`'s standard convention. As noted above, `thop` may under-count custom attention math implemented with raw tensor ops (it reliably counts `nn.Linear`/`nn.Conv1d`, which dominate PatchTST's parameter count); treat these as relative, not absolute, figures.

In [16]:
from thop import profile

single_x = sample_x[:1]
single_y = sample_y[:1]
single_x_mark = sample_x_mark[:1]
single_y_mark = sample_y_mark[:1]

flops_counts = {}

for name, m in {"B4_no_pruning": baseline_model, **pruned_models}.items():
    m_copy = copy.deepcopy(m)
    flops, _ = profile(m_copy, inputs=(single_x, single_x_mark, single_y, single_y_mark), verbose=False)
    flops_counts[name] = flops
    del m_copy

flops_df = pd.DataFrame([
    {"setting": k, "flops": v, "flops_vs_baseline_percent": (v / flops_counts["B4_no_pruning"] - 1) * 100}
    for k, v in flops_counts.items()
])
display(flops_df)

,setting,flops,flops_vs_baseline_percent
0,B4_no_pruning,120873984.0,0.000000
1,static_25,106423296.0,-11.955168
2,joint_trend,106423296.0,-11.955168
3,joint_seasonal,106423296.0,-11.955168
4,joint_residual,106423296.0,-11.955168


## 15. Inference latency

Wall-clock GPU time for a realistic batch (32 windows, the batch size used throughout training/evaluation), averaged over 50 timed runs after a 10-run warm-up (to let CUDA kernels/caches settle).

In [17]:
def benchmark_latency(model, inputs, device, warmup=10, repeats=50):
    model.eval()
    sx, sxm, sy, sym = inputs

    with torch.no_grad():
        for _ in range(warmup):
            _ = model(sx, sxm, sy, sym)
        if device.type == "cuda":
            torch.cuda.synchronize()

        times_ms = []
        for _ in range(repeats):
            start = time.perf_counter()
            _ = model(sx, sxm, sy, sym)
            if device.type == "cuda":
                torch.cuda.synchronize()
            times_ms.append((time.perf_counter() - start) * 1000)

    times_ms = np.array(times_ms)
    return {
        "mean_ms_per_batch": float(times_ms.mean()),
        "std_ms_per_batch": float(times_ms.std()),
        "mean_ms_per_sample": float(times_ms.mean() / sx.shape[0]),
        "batch_size": int(sx.shape[0]),
    }


latency_inputs = (sample_x, sample_x_mark, sample_y, sample_y_mark)

latency_results = {"B4_no_pruning": benchmark_latency(baseline_model, latency_inputs, device)}
for name, m in pruned_models.items():
    latency_results[name] = benchmark_latency(m, latency_inputs, device)

latency_df = pd.DataFrame([{"setting": k, **v} for k, v in latency_results.items()])
baseline_latency = latency_df.loc[latency_df["setting"] == "B4_no_pruning", "mean_ms_per_batch"].iloc[0]
latency_df["latency_vs_baseline_percent"] = (latency_df["mean_ms_per_batch"] / baseline_latency - 1) * 100
display(latency_df)

,setting,mean_ms_per_batch,std_ms_per_batch,mean_ms_per_sample,batch_size,latency_vs_baseline_percent
0,B4_no_pruning,8.979411,2.936011,0.280607,32,0.000000
1,static_25,8.906491,2.839566,0.278328,32,-0.812085
2,joint_trend,9.357740,3.648952,0.292429,32,4.213294
3,joint_seasonal,12.427299,5.666167,0.388353,32,38.397702
4,joint_residual,13.718944,4.913636,0.428717,32,52.782222


## 16. Policy B's expected cost (regime-frequency-weighted) and regime-detection overhead

A deployment of the joint dynamic method would route each window to the structurally pruned sub-model matching its detected regime. This section combines the three regime-specific models' costs into a single "expected per-window" figure weighted by how often each regime actually occurs (using the test-set distribution from Experiment 07/10: ~93.6% trend, ~4.5% seasonal, ~1.9% residual), and separately measures the wall-clock cost of the STL regime-detection step itself — a real cost Policy B pays per window that Policy A (static pruning) and the unpruned baseline do not.

In [18]:
def weighted_policy_b_cost(metric_dict_by_regime, frequency):
    return sum(metric_dict_by_regime[r] * frequency[r] for r in frequency)

policy_b_expected_params = weighted_policy_b_cost(
    {r: param_counts[f"joint_{r}"] for r in TEST_REGIME_FREQUENCY}, TEST_REGIME_FREQUENCY
)
policy_b_expected_flops = weighted_policy_b_cost(
    {r: flops_counts[f"joint_{r}"] for r in TEST_REGIME_FREQUENCY}, TEST_REGIME_FREQUENCY
)
policy_b_expected_latency = weighted_policy_b_cost(
    {r: latency_results[f"joint_{r}"]["mean_ms_per_batch"] for r in TEST_REGIME_FREQUENCY}, TEST_REGIME_FREQUENCY
)

print("Policy B (joint dynamic) regime-frequency-weighted expected parameters:", policy_b_expected_params)
print("Policy B (joint dynamic) regime-frequency-weighted expected FLOPs:", policy_b_expected_flops)
print("Policy B (joint dynamic) regime-frequency-weighted expected latency (ms/batch):", policy_b_expected_latency)

Policy B (joint dynamic) regime-frequency-weighted expected parameters: 866495.9999999999
Policy B (joint dynamic) regime-frequency-weighted expected FLOPs: 106423296.0
Policy B (joint dynamic) regime-frequency-weighted expected latency (ms/batch): 9.580074309521542


In [19]:
from statsmodels.tsa.seasonal import STL

def label_window_with_stl(series_window, period=24):
    result = STL(series_window, period=period, robust=True).fit()
    return result.trend, result.seasonal, result.resid


regime_path = REGIME_DIR / "etth1_validation_regimes_ot_seq336.csv"
regime_df = pd.read_csv(regime_path)

df = pd.read_csv(tslib_data_path)
TRAIN_SIZE = 12 * 30 * 24
VAL_SIZE = 4 * 30 * 24
SEQ_LEN = 336
val_border1 = TRAIN_SIZE - SEQ_LEN
val_border2 = TRAIN_SIZE + VAL_SIZE
validation_segment = df.iloc[val_border1:val_border2].reset_index(drop=True)
target_series = validation_segment["OT"].astype(float).to_numpy()

STL_BENCHMARK_WINDOWS = 100
stl_times_ms = []
for window_id in range(STL_BENCHMARK_WINDOWS):
    window = target_series[window_id:window_id + SEQ_LEN]
    start = time.perf_counter()
    label_window_with_stl(window, period=24)
    stl_times_ms.append((time.perf_counter() - start) * 1000)

stl_times_ms = np.array(stl_times_ms)
print(f"STL regime detection: {stl_times_ms.mean():.2f} +/- {stl_times_ms.std():.2f} ms/window "
      f"(CPU, statsmodels, over {STL_BENCHMARK_WINDOWS} windows)")
print("This is a per-window, one-time cost that Policy B (dynamic) pays and Policy A / the unpruned baseline do not.")

STL regime detection: 83.35 +/- 51.75 ms/window (CPU, statsmodels, over 100 windows)
This is a per-window, one-time cost that Policy B (dynamic) pays and Policy A / the unpruned baseline do not.


## 17. Consolidated efficiency table

In [20]:
efficiency_df = param_df.merge(flops_df, on="setting").merge(
    latency_df[["setting", "mean_ms_per_batch", "mean_ms_per_sample", "latency_vs_baseline_percent"]],
    on="setting",
)

policy_b_row = pd.DataFrame([{
    "setting": "joint_dynamic_expected_weighted",
    "parameters": policy_b_expected_params,
    "parameters_vs_baseline_percent": (policy_b_expected_params / param_counts["B4_no_pruning"] - 1) * 100,
    "flops": policy_b_expected_flops,
    "flops_vs_baseline_percent": (policy_b_expected_flops / flops_counts["B4_no_pruning"] - 1) * 100,
    "mean_ms_per_batch": policy_b_expected_latency,
    "mean_ms_per_sample": policy_b_expected_latency / latency_results["B4_no_pruning"]["batch_size"],
    "latency_vs_baseline_percent": (policy_b_expected_latency / baseline_latency - 1) * 100,
}])

efficiency_df = pd.concat([efficiency_df, policy_b_row], ignore_index=True)
display(efficiency_df)

,setting,parameters,parameters_vs_baseline_percent,flops,flops_vs_baseline_percent,mean_ms_per_batch,mean_ms_per_sample,latency_vs_baseline_percent
0,B4_no_pruning,915936.0,0.000000,120873984.0,0.000000,8.979411,0.280607,0.000000
1,static_25,866496.0,-5.397757,106423296.0,-11.955168,8.906491,0.278328,-0.812085
2,joint_trend,866496.0,-5.397757,106423296.0,-11.955168,9.357740,0.292429,4.213294
3,joint_seasonal,866496.0,-5.397757,106423296.0,-11.955168,12.427299,0.388353,38.397702
4,joint_residual,866496.0,-5.397757,106423296.0,-11.955168,13.718944,0.428717,52.782222
5,joint_dynamic_expected_weighted,866496.0,-5.397757,106423296.0,-11.955168,9.580074,0.299377,6.689335


## 18. Save results

In [21]:
efficiency_df.to_csv(EFFICIENCY_DIR / "efficiency_metrics.csv", index=False)

stl_overhead_df = pd.DataFrame([{
    "mean_ms_per_window": float(stl_times_ms.mean()),
    "std_ms_per_window": float(stl_times_ms.std()),
    "num_windows_benchmarked": STL_BENCHMARK_WINDOWS,
}])
stl_overhead_df.to_csv(EFFICIENCY_DIR / "stl_regime_detection_overhead.csv", index=False)

keep_heads_by_setting = {"static_25": keep_heads_static}
keep_heads_by_setting.update({f"joint_{r}": kh for r, kh in keep_heads_joint.items()})

for name, kh in keep_heads_by_setting.items():
    pd.DataFrame({"layer": range(num_layers), "keep_heads": [str(layer_heads) for layer_heads in kh]}).to_csv(
        EFFICIENCY_DIR / f"{name}_keep_heads_per_layer.csv", index=False
    )

print("Saved Experiment 11 outputs to:", EFFICIENCY_DIR)
for path in sorted(EFFICIENCY_DIR.iterdir()):
    print(" -", path.name)

Saved Experiment 11 outputs to: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/b4_efficiency_metrics
 - efficiency_metrics.csv
 - joint_residual_keep_heads_per_layer.csv
 - joint_seasonal_keep_heads_per_layer.csv
 - joint_trend_keep_heads_per_layer.csv
 - static_25_keep_heads_per_layer.csv
 - stl_regime_detection_overhead.csv
